# Practical 7 — Identify a Webpage Using NLTK

## Objective
Fetch a webpage, extract readable text, preprocess it with NLTK, and infer the webpage topic from frequent meaningful words.

**How to use:** Put the required CSV file(s) in the `data/` folder, update the filename/path if needed, and run cells from top to bottom. Where a dataset is unavailable, the notebook reports that clearly rather than inventing results.

## Imports

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

DATA_DIR = Path("../data")
DATA_DIR.mkdir(exist_ok=True)

## Install/download NLTK resources if needed

In [ ]:
# If packages/resources are missing, run in a terminal:
# pip install requests beautifulsoup4 nltk
import re
import requests
from bs4 import BeautifulSoup
import nltk
for resource in ["punkt", "stopwords"]:
    try:
        nltk.data.find(f"tokenizers/{resource}" if resource == "punkt" else f"corpora/{resource}")
    except LookupError:
        nltk.download(resource)
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize

In [ ]:
url = input("Enter webpage URL (including https://): ").strip()
if not url.startswith(("http://", "https://")):
    raise ValueError("Enter a complete URL beginning with http:// or https://.")
response = requests.get(url, timeout=20, headers={"User-Agent": "Mozilla/5.0 (educational NLP lab)"})
response.raise_for_status()
soup = BeautifulSoup(response.text, "html.parser")
for element in soup(["script", "style", "nav", "footer", "header", "noscript"]):
    element.decompose()
page_title = soup.title.get_text(" ", strip=True) if soup.title else "(No title found)"
page_text = soup.get_text(" ", strip=True)
print("Page title:", page_title)
print("Extracted text preview:", page_text[:1000])

In [ ]:
tokens = word_tokenize(page_text.lower())
stop_words = set(stopwords.words("english"))
words = [re.sub(r"[^a-z]", "", t) for t in tokens]
words = [w for w in words if len(w) > 2 and w not in stop_words]
frequency = pd.Series(words).value_counts().head(20)
display(frequency.to_frame("count"))
print("Likely topic keywords:", ", ".join(frequency.head(10).index.tolist()))
print("Topic inference is heuristic: verify the keywords and title rather than treating this as a trained classifier.")

## Conclusion

State the webpage title and infer its likely subject using the extracted content and frequent keywords. Some websites block automated requests or render content using JavaScript.